# Building Model

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("data/cleaned_property_data_v2.csv")
df.head()

,type,area_in_marla,bedroom,bath,location_city,price
0,House,20.0,7.0,6.0,Islamabad,190000000.0
1,House,14.2,6.0,6.0,Islamabad,60000000.0
2,House,20.0,8.0,7.0,Islamabad,70000000.0
3,House,8.0,4.0,6.0,Islamabad,26500000.0
4,Flat,2.4,1.0,1.0,Islamabad,4000000.0


First we need to convert the type and location_city columns to numbers. We are gonna use OHE for this.

---

## OneHotEncoder

In [3]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)

encoded = ohe.fit_transform(df[["type", "location_city"]])

In [4]:
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(["type", "location_city"]))

encoded_df.head()

,type_Agricultural Land,type_Building,type_Commercial Plot,type_Flat,type_House,type_Office,type_Other,type_Plot File,type_Residential Plot,type_Shop,...,location_city_Karachi,location_city_Lahore,location_city_Multan,location_city_Other,location_city_Peshawar,location_city_Quetta,location_city_Rawalpindi,location_city_Sialkot,location_city_Sukkur,location_city_Wah
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df = pd.concat([df, encoded_df], axis=1)
df.head(2)

,type,area_in_marla,bedroom,bath,location_city,price,type_Agricultural Land,type_Building,type_Commercial Plot,type_Flat,...,location_city_Karachi,location_city_Lahore,location_city_Multan,location_city_Other,location_city_Peshawar,location_city_Quetta,location_city_Rawalpindi,location_city_Sialkot,location_city_Sukkur,location_city_Wah
0,House,20.0,7.0,6.0,Islamabad,190000000.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,House,14.2,6.0,6.0,Islamabad,60000000.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Now we will remove the type and location_city column

In [6]:
df = df.drop(columns = ["type", "location_city"])
df["price"] = df.pop("price")
df.head(2)

,area_in_marla,bedroom,bath,type_Agricultural Land,type_Building,type_Commercial Plot,type_Flat,type_House,type_Office,type_Other,...,location_city_Lahore,location_city_Multan,location_city_Other,location_city_Peshawar,location_city_Quetta,location_city_Rawalpindi,location_city_Sialkot,location_city_Sukkur,location_city_Wah,price
0,20.0,7.0,6.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,190000000.0
1,14.2,6.0,6.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60000000.0


---

As we had seen in the visualization notebook, we are gonna use log transformation on our price column.

In [7]:
df["price"] = np.log1p(df["price"])
X = df.drop(columns = "price")
y = df["price"]

---

## Train Test Split

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

print(X_train.shape, X_test.shape)

(67981, 30) (16996, 30)


---

## Using RandomizedSearchCV

We are gonna use RSCV to find the best model and paramters to train our model.

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

We are using regressor as it predicts a name, where a classifer predicts a category(spam or not spam, etc).

In [10]:
parameters = {
    "linearReg" : {"model": LinearRegression(), "params": {}  # no parameters to tune
    },

    "decisionTree": {"model": DecisionTreeRegressor(), "params": {"max_depth": [3, 5, 10, None], "min_samples_split": [2, 5, 10]}
    },

    "randomForest": {"model": RandomForestRegressor(), "params": {"n_estimators": [50, 100, 200], "max_depth": [5, 10, None]}
    }
}

scores = []

for model_name, model_params in parameters.items():
    randSCV = RandomizedSearchCV(model_params["model"], model_params["params"], cv = 5, 
                                 random_state = 42, n_iter = 10, n_jobs = -1, scoring = "r2")
    # n_jobs = -1 : how many cpu cores to use...-1 means use all the available cores, for faster task completion.
    # scoring = "r2" : it is the metric used to compare combinations and decide which is best. r^2 measures how much of the var in price your model explains
    randSCV.fit(X_train, y_train)
    scores.append({
        "model": model_name,
        "best_score": randSCV.best_score_,
        "best_params": randSCV.best_params_
    })
scores_df = pd.DataFrame(scores)

c:\Users\Ahmad Maqsood\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\Ahmad Maqsood\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 9 is smaller than n_iter=10. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [12]:
scores_df

,model,best_score,best_params
0,linearReg,0.543890,{}
1,decisionTree,0.773441,"{'min_samples_split': 10, 'max_depth': None}"
2,randomForest,0.786848,"{'n_estimators': 200, 'max_depth': None}"


Ok, so randomforestregressor is performing best for this data. We also have the best paramters, let's train the model.

---

## RandomForestRegressor

In [13]:
model = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

### Model Score

In [14]:
print(f"Accuracy Score of the model is : {model.score(X_test, y_test)}")

Accuracy Score of the model is : 0.7899855220435377


---

## Saving the Model & OHE

In [22]:
import joblib
joblib.dump(model, "model/random_forest_model.pkl")
joblib.dump(ohe, "model/ohe_encoder.pkl")

['model/ohe_encoder.pkl']